# Лабораторная работа 11. Online softmax и FlashAttention (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 4:** инференс и эффективность

**Цель.** Разобрать алгоритм точного attention, который не хранит матрицу $T \times T$ и потому быстрее наивной реализации на GPU.

**Что нужно сдать:**
1. Online softmax и блочное attention на PyTorch, численно эквивалентные обычному (FP32, FP16, BF16).
2. Прямой проход FlashAttention на Triton (каузальная маска, тайлинг по $Q$ и $K/V$).
3. Бенчмарк скорости и пиковой памяти: наивная реализация, блочная, `scaled_dot_product_attention`, Triton.
4. Roofline-оценку для T4 и объяснение результатов.
5. Дополнительно: обратный проход с пересчётом и проверкой градиентов.

**Вычисления.** Пункты 2–3 с Triton требуют GPU (Colab T4). На CPU ноутбук выполняет всё остальное и пропускает ячейки Triton.

**Теория. Online softmax.**
Для блоков $x^{(1)}, x^{(2)}, \dots$ поддерживаются текущий максимум $m$ и нормировка $\ell$:

$$m^{\text{new}} = \max\left(m, \max_j x^{(b)}_j\right), \qquad \ell^{\text{new}} = e^{m - m^{\text{new}}}\ell + \sum_j e^{x^{(b)}_j - m^{\text{new}}}$$

По индукции $\ell = \sum_{j \le \text{текущий}} e^{x_j - m}$, поэтому в конце $\mathrm{softmax}(x)_j = e^{x_j - m}/\ell$.

Для attention вместе с $\ell$ так же пересчитывается ненормированный выход:

$$o^{\text{new}} = e^{m - m^{\text{new}}}\,o + \sum_j e^{s_j - m^{\text{new}}}\,v_j, \qquad O = \frac{o}{\ell}$$


In [1]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

torch 2.12.0.dev20260408+cu128, device=cuda, FAST_DEV_RUN=True


## 1. Online softmax для вектора

In [ ]:
def online_softmax(x, block=16):
    """Двухпроходный алгоритм: (1) потоковые m и l по блокам; (2) нормировка."""
    flat = x.reshape(-1)
    n = flat.numel()
    m = torch.tensor(float("-inf"), dtype=torch.float32, device=x.device)
    l = torch.tensor(0.0,           dtype=torch.float32, device=x.device)

    for i in range(0, n, block):
        xb = flat[i:i+block].float()
        mb = xb.max()
        m_new = torch.maximum(m, mb)

        alpha = torch.exp(m - m_new)
        alpha = torch.where(torch.isfinite(alpha), alpha, torch.zeros_like(alpha))
        l = alpha * l + torch.exp(xb - m_new).sum()
        m = m_new

    out = torch.empty_like(flat, dtype=torch.float32)
    for i in range(0, n, block):
        out[i:i+block] = torch.exp(flat[i:i+block].float() - m) / l
    return out.to(x.dtype).view(x.shape)

x = torch.randn(1000) * 20
assert torch.allclose(online_softmax(x), x.softmax(0), atol=1e-6)
x = torch.tensor([1000.0, 1001.0, -1000.0])                     # устойчивость к переполнению
assert torch.allclose(online_softmax(x, block=1), x.softmax(0))
print("online softmax совпадает с torch.softmax")

online softmax совпадает с torch.softmax


## 2. Блочное attention (FlashAttention-подобный прямой проход на PyTorch)

Входы `q, k, v` формы `(B, H, T, d)`. Внешний цикл — по блокам запросов размера `Br`, внутренний — по блокам ключей `Bc`. Аккумуляторы \(m, \ell, o\) держите в FP32 независимо от типа входа. При каузальной маске пропускайте блоки ключей, целиком лежащие правее блока запросов. Функция дополнительно возвращает logsumexp \(L = m + \log\ell\) — он понадобится для обратного прохода.

In [ ]:
def naive_attention(q, k, v, causal=False):
    s = (q @ k.transpose(-1, -2)).float() / math.sqrt(q.size(-1))
    if causal: s = s.masked_fill(torch.ones(s.shape[-2:], dtype=torch.bool, device=q.device).triu(1), float("-inf"))
    return (s.softmax(-1) @ v.float()).to(q.dtype)

def blocked_attention(q, k, v, causal=False, Br=64, Bc=64):
    B, H, T, d = q.shape; scale = 1 / math.sqrt(d)
    O = torch.empty_like(q); LSE = torch.empty(B, H, T, dtype=torch.float32, device=q.device)
    
    for b in range(B):
        for h in range(H):
            q_bh = q[b, h].float()
            k_bh = k[b, h].float()
            v_bh = v[b, h].float()

            for i in range(0, T, Br):
                qi = q_bh[i:i+Br]
                Br_i = qi.size(0)
                m   = torch.full((Br_i,), float("-inf"), device=q.device)
                l   = torch.zeros(Br_i, device=q.device)
                acc = torch.zeros(Br_i, d, device=q.device)

                j_end = min(T, i + Br_i) if causal else T

                for j in range(0, j_end, Bc):
                    kj = k_bh[j:j+Bc]; vj = v_bh[j:j+Bc]
                    Bc_j = kj.size(0)
                    s = qi @ kj.T * scale

                    if causal:
                        row = torch.arange(i, i+Br_i, device=q.device).unsqueeze(1)
                        col = torch.arange(j, j+Bc_j, device=q.device).unsqueeze(0)
                        s = s.masked_fill(col > row, float("-inf"))

                    m_new = torch.maximum(m, s.max(dim=-1).values)
                    
                    alpha = torch.exp(m - m_new)
                    alpha = torch.where(torch.isfinite(alpha), alpha, torch.zeros_like(alpha))

                    p = torch.exp(s - m_new.unsqueeze(-1))
                    p = torch.where(torch.isfinite(p), p, torch.zeros_like(p))

                    l   = alpha * l + p.sum(dim=-1)
                    acc = alpha.unsqueeze(-1) * acc + p @ vj
                    m   = m_new

                O[b, h, i:i+Br_i] = (acc / l.unsqueeze(-1)).to(q.dtype)
                LSE[b, h, i:i+Br_i] = m + torch.log(l)
    return O, LSE

torch.manual_seed(0)
for dtype, tol in [(torch.float32, 1e-5), (torch.bfloat16, 2e-2), (torch.float16, 5e-3)]:
    q, k, v = (torch.randn(2, 3, 200, 64).to(dtype) for _ in range(3))
    for causal in (False, True):
        ref = F.scaled_dot_product_attention(q.float(), k.float(), v.float(), is_causal=causal)
        out, _ = blocked_attention(q, k, v, causal, Br=48, Bc=32)
        err = (out.float() - ref).abs().max().item()
        print(f"{str(dtype):15s} causal={causal}: max|Δ| = {err:.2e}"); assert err < tol

torch.float32   causal=False: max|Δ| = 3.58e-07
torch.float32   causal=True: max|Δ| = 3.58e-07
torch.bfloat16  causal=False: max|Δ| = 1.84e-03
torch.bfloat16  causal=True: max|Δ| = 4.41e-03
torch.float16   causal=False: max|Δ| = 2.09e-04
torch.float16   causal=True: max|Δ| = 9.46e-04


## 3. FlashAttention на Triton (GPU)

Реализуйте ядро прямого прохода. Каждая программа обрабатывает один блок из `BLOCK_M` запросов для одной пары (batch, head):
1. загрузить блок \(Q\);
2. в цикле по блокам \(K, V\) размера `BLOCK_N` вычислить \(S = QK^\top\cdot\text{scale}\), применить маски (выход за длину и каузальную), обновить \(m, \ell\), аккумулятор;
3. записать \(O = \text{acc}/\ell\).

Для каузальной маски цикл идёт только до блока, содержащего диагональ.

In [ ]:
import platform
import subprocess
import sys

HAS_TRITON = False

if DEVICE == "cuda":
    try:
        import triton, triton.language as tl
        HAS_TRITON = True
    except ImportError:
        pkg = "triton-windows" if platform.system() == "Windows" else "triton"
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])
            import triton, triton.language as tl
            HAS_TRITON = True
        except Exception as e:
            print(f"Не удалось установить {pkg}: {e}")

print("Triton доступен:", HAS_TRITON)


Triton доступен: True


In [10]:
if HAS_TRITON:
    @triton.jit
    def flash_fwd_kernel(Q, K, V, O, sm_scale, T, stride_bh, stride_t,
                         BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
                         D: tl.constexpr, CAUSAL: tl.constexpr):
        pid_m  = tl.program_id(0)
        pid_bh = tl.program_id(1)
        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_d = tl.arange(0, D)
        base   = pid_bh * stride_bh

        m_mask = offs_m < T
        q = tl.load(Q + base + offs_m[:, None] * stride_t + offs_d[None, :],
                    mask=m_mask[:, None], other=0.0)

        m_i = tl.full([BLOCK_M], float("-inf"), dtype=tl.float32)
        l_i = tl.zeros([BLOCK_M], dtype=tl.float32)
        acc = tl.zeros([BLOCK_M, D], dtype=tl.float32)

        hi = T
        if CAUSAL:
            hi = tl.minimum(T, (pid_m + 1) * BLOCK_M)

        for start_n in range(0, hi, BLOCK_N):
            offs_n = start_n + tl.arange(0, BLOCK_N)
            n_mask = offs_n < T

            k = tl.load(K + base + offs_n[:, None] * stride_t + offs_d[None, :],
                        mask=n_mask[:, None], other=0.0)
            v = tl.load(V + base + offs_n[:, None] * stride_t + offs_d[None, :],
                        mask=n_mask[:, None], other=0.0)

            s = tl.dot(q, tl.trans(k)) * sm_scale

            if CAUSAL:
                s = tl.where(offs_m[:, None] >= offs_n[None, :], s, float("-inf"))
            s = tl.where(n_mask[None, :], s, float("-inf"))

            m_ij   = tl.max(s, axis=1)
            m_new  = tl.maximum(m_i, m_ij)
            alpha  = tl.exp(m_i - m_new)
            alpha  = tl.where(m_new == float("-inf"), 0.0, alpha)

            p = tl.exp(s - m_new[:, None])
            p = tl.where(m_new[:, None] == float("-inf"), 0.0, p)

            l_i = alpha * l_i + tl.sum(p, axis=1)
            acc = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v)
            m_i = m_new

        acc = acc / l_i[:, None]
        tl.store(O + base + offs_m[:, None] * stride_t + offs_d[None, :],
                 acc.to(O.dtype.element_ty), mask=m_mask[:, None])

    def flash_attention_triton(q, k, v, causal=False, BLOCK_M=64, BLOCK_N=64):
        B, H, T, D = q.shape
        assert D in (16, 32, 64, 128) and q.dtype in (torch.float16, torch.bfloat16)
        q3, k3, v3 = (t.contiguous().view(B * H, T, D) for t in (q, k, v)); o = torch.empty_like(q3)
        grid = (triton.cdiv(T, BLOCK_M), B * H)
        flash_fwd_kernel[grid](q3, k3, v3, o, 1 / math.sqrt(D), T, T * D, D, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, D=D, CAUSAL=causal)
        return o.view(B, H, T, D)

    for causal in (False, True):
        q, k, v = (torch.randn(2, 4, 1000, 64, device="cuda", dtype=torch.float16) for _ in range(3))
        ref = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        err = (flash_attention_triton(q, k, v, causal) - ref).abs().max().item()
        print(f"Triton causal={causal}: max|Δ| = {err:.2e}"); assert err < 1e-2
else:
    print("GPU нет — ячейка Triton пропущена. Запустите ноутбук в Colab с GPU T4.")

Triton causal=False: max|Δ| = 1.22e-04
Triton causal=True: max|Δ| = 2.44e-04


## 4. Бенчмарк скорости и памяти

Для длин \(T\) от 512 до 8192 (на CPU — до 2048), \(d = 64\), \(B\cdot H = 8\) измерьте время прямого прохода и пиковую память (на GPU — `torch.cuda.max_memory_allocated`; на CPU — оценка по размеру промежуточных тензоров) для:
- наивной реализации;
- блочной PyTorch-реализации;
- `F.scaled_dot_product_attention`;
- Triton-ядра (на GPU).

Объясните, почему блочная реализация на PyTorch медленнее наивной, хотя использует тот же алгоритм, что и FlashAttention.

In [12]:
def bench(fn, reps=3):
    if DEVICE == "cuda": torch.cuda.synchronize()
    fn()
    if DEVICE == "cuda": torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(reps): fn()
    if DEVICE == "cuda": torch.cuda.synchronize()
    mem = torch.cuda.max_memory_allocated() / 2 ** 20 if DEVICE == "cuda" else float("nan")
    return (time.time() - t0) / reps * 1000, mem

LENGTHS = [256, 512, 1024, 2048] if DEVICE == "cpu" else [512, 1024, 2048, 4096, 8192]
dtype = torch.float16 if DEVICE == "cuda" else torch.float32
B, H, d = 2, 4, 64  
rows = []

# (заполните rows для всех методов и длин; для CPU добавьте оценку памяти naive = B*H*T*T*4 байт)
with torch.no_grad():
    for T in LENGTHS:
        q, k, v = (torch.randn(B, H, T, d, device=DEVICE, dtype=dtype) for _ in range(3))

        t, mem = bench(lambda: naive_attention(q, k, v, causal=True))
        if DEVICE == "cpu":
            mem = 2 * B * H * T * T * 4 / 2**20
        rows.append(("naive", T, t, mem))

        t, mem = bench(lambda: blocked_attention(q, k, v, causal=True)[0])
        if DEVICE == "cpu":
            mem = 4 * B * H * T * d * 4 / 2**20
        rows.append(("blocked", T, t, mem))

        t, mem = bench(lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True))
        if DEVICE == "cpu":
            mem = 4 * B * H * T * d * 4 / 2**20
        rows.append(("sdpa", T, t, mem))

        if HAS_TRITON:
            q16, k16, v16 = q.to(torch.float16), k.to(torch.float16), v.to(torch.float16)
            t, mem = bench(lambda: flash_attention_triton(q16, k16, v16, causal=True))
            rows.append(("triton", T, t, mem))

print(f"{'method':10s} {'T':>6s} {'time, ms':>10s} {'mem, MB':>9s}")
for name, T, t, mem in rows:
    print(f"{name:10s} {T:>6d} {t:>10.2f} {mem:>9.2f}")

method          T   time, ms   mem, MB
naive         512       0.33     63.22
blocked       512     149.31     46.53
sdpa          512       0.00     45.72
triton        512       0.00     38.88
naive        1024       0.86    109.38
blocked      1024     547.14     43.96
sdpa         1024       0.00     42.38
triton       1024       0.00     42.38
naive        2048       3.36    311.38
blocked      2048    2176.99     52.49
sdpa         2048       0.33     49.38
triton       2048       0.00     46.38
naive        4096      12.24   1096.38
blocked      4096    7866.59     66.56
sdpa         4096       1.00     60.38
triton       4096       0.67     54.38
naive        8192      50.82   4234.38
blocked      8192   32999.92     94.68
sdpa         8192       2.85     82.38
triton       8192       1.17     70.38


## 5. Roofline-оценка для T4

Параметры T4: пиковая производительность FP16 на тензорных ядрах около 65 TFLOP/s, пропускная способность памяти около 320 ГБ/с. Для одной головы с длиной \(T\) и \(d = 64\) в FP16 оцените:
1. FLOPs прямого прохода (\(4T^2d\) без учёта маски);
2. объём обмена с HBM для наивной реализации (чтение \(Q, K, V\), запись и чтение \(S\) и \(P\), запись \(O\));
3. то же для FlashAttention (\(Q\) и \(O\) — один раз, \(K, V\) — \(T/B_r\) раз);
4. арифметическую интенсивность и ограничение (compute-bound или memory-bound) в обоих случаях, сравнив с точкой перегиба \(65\cdot10^{12} / 320\cdot10^{9} \approx 200\) FLOP/байт.

In [13]:
def roofline(T, d=64, Br=64, bytes_per=2, peak=65e12, bw=320e9):
    ridge = peak / bw
    flops = 4 * T * T * d

    naive_bytes = (3 * T * d + T * d) * bytes_per + 4 * T * T * bytes_per

    flash_bytes = 2 * T * d * bytes_per + 2 * (T / Br) * T * d * bytes_per

    def pack(fl, by):
        ai = fl / by
        return {"flops": fl, "bytes": by, "ai": ai,
                "bound": "compute" if ai > ridge else "memory",
                "time_us_roof": max(fl / peak, by / bw) * 1e6}

    return {"naive": pack(flops, naive_bytes),
            "flash": pack(flops, flash_bytes)}

for T in (1024, 4096, 16384):
    print(T, {k: {kk: (round(vv, 1) if isinstance(vv, float) else vv) for kk, vv in v.items()} for k, v in roofline(T).items()})

1024 {'naive': {'flops': 268435456, 'bytes': 8912896, 'ai': 30.1, 'bound': 'memory', 'time_us_roof': 27.9}, 'flash': {'flops': 268435456, 'bytes': 4456448.0, 'ai': 60.2, 'bound': 'memory', 'time_us_roof': 13.9}}
4096 {'naive': {'flops': 4294967296, 'bytes': 136314880, 'ai': 31.5, 'bound': 'memory', 'time_us_roof': 426.0}, 'flash': {'flops': 4294967296, 'bytes': 68157440.0, 'ai': 63.0, 'bound': 'memory', 'time_us_roof': 213.0}}
16384 {'naive': {'flops': 68719476736, 'bytes': 2155872256, 'ai': 31.9, 'bound': 'memory', 'time_us_roof': 6737.1}, 'flash': {'flops': 68719476736, 'bytes': 1077936128.0, 'ai': 63.8, 'bound': 'memory', 'time_us_roof': 3368.6}}


## 6. Дополнительно: обратный проход с пересчётом

Реализуйте `torch.autograd.Function`, где `forward` вызывает `blocked_attention` и сохраняет только $Q, K, V, O$ и $L$ (без $P$). В `backward` для каждого блока пересчитайте $P_{ij} = \exp(S_{ij} - L_i)$ и накопите

$$
dV_j \mathrel{+}= P_{ij}^\top dO_i, \quad dP_{ij} = dO_i V_j^\top, \quad dS_{ij} = P_{ij} \odot (dP_{ij} - D_i), \quad dQ_i \mathrel{+}= \text{scale} \cdot dS_{ij} K_j, \quad dK_j \mathrel{+}= \text{scale} \cdot dS_{ij}^\top Q_i,
$$

где $D_i = \mathrm{rowsum}(dO_i \odot O_i)$. Сверьте градиенты с autograd.


In [ ]:
class FlashAttnFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, causal):
        O, L = blocked_attention(q, k, v, causal)
        ctx.save_for_backward(q, k, v, O, L)
        ctx.causal = causal
        return O

    @staticmethod
    def backward(ctx, dO):
        q, k, v, O, L = ctx.saved_tensors
        causal = ctx.causal
        B, H, T, d = q.shape
        scale = 1 / math.sqrt(d)
        Br = Bc = 64

        cdt = torch.float32 if q.dtype in (torch.float16, torch.bfloat16) else q.dtype
        dq = torch.zeros_like(q); dk = torch.zeros_like(k); dv = torch.zeros_like(v)
        D  = (dO * O).sum(-1)

        for b in range(B):
            for h in range(H):
                q_bh, k_bh, v_bh = q[b,h].to(cdt), k[b,h].to(cdt), v[b,h].to(cdt)
                dO_bh, L_bh, D_bh = dO[b,h].to(cdt), L[b,h], D[b,h]

                dq_bh = torch.zeros(T, d, device=q.device, dtype=cdt)
                dk_bh = torch.zeros(T, d, device=q.device, dtype=cdt)
                dv_bh = torch.zeros(T, d, device=q.device, dtype=cdt)

                for i in range(0, T, Br):
                    qi = q_bh[i:i+Br];  dOi = dO_bh[i:i+Br]
                    Li = L_bh[i:i+Br];  Di  = D_bh[i:i+Br]
                    Br_i = qi.size(0)
                    dqi  = torch.zeros(Br_i, d, device=q.device, dtype=cdt)

                    j_end = min(T, i + Br_i) if causal else T
                    for j in range(0, j_end, Bc):
                        kj = k_bh[j:j+Bc]; vj = v_bh[j:j+Bc]
                        Bc_j = kj.size(0)
                        s = qi @ kj.T * scale
                        if causal:
                            row = torch.arange(i, i+Br_i, device=q.device).unsqueeze(1)
                            col = torch.arange(j, j+Bc_j, device=q.device).unsqueeze(0)
                            s = s.masked_fill(col > row, float("-inf"))

                        p = torch.exp(s - Li.unsqueeze(-1))
                        p = torch.where(torch.isfinite(p), p, torch.zeros_like(p))

                        dp = dOi @ vj.T
                        ds = p * (dp - Di.unsqueeze(-1))

                        dv_bh[j:j+Bc_j] += p.T @ dOi
                        dqi             += ds @ kj * scale
                        dk_bh[j:j+Bc_j] += ds.T @ qi * scale

                    dq_bh[i:i+Br_i] = dqi

                dq[b,h] = dq_bh.to(dq.dtype)
                dk[b,h] = dk_bh.to(dk.dtype)
                dv[b,h] = dv_bh.to(dv.dtype)

        return dq, dk, dv, None

q, k, v = (torch.randn(1, 2, 150, 32, dtype=torch.float64, requires_grad=True) for _ in range(3))
for causal in (False, True):
    dO = torch.randn(1, 2, 150, 32, dtype=torch.float64)
    g1 = torch.autograd.grad((FlashAttnFn.apply(q, k, v, causal) * dO).sum(), (q, k, v))
    g2 = torch.autograd.grad((naive_attention(q, k, v, causal) * dO).sum(), (q, k, v))
    print(f"causal={causal}:", [f"{(a - b).abs().max().item():.1e}" for a, b in zip(g1, g2)])
    assert all(torch.allclose(a, b, atol=1e-5) for a, b in zip(g1, g2))   # прямой проход накапливает в FP32

causal=False: ['3.0e-07', '3.0e-07', '2.7e-07']
causal=True: ['4.3e-07', '5.9e-07', '1.5e-06']


## Вопросы

1. Докажите корректность online softmax по индукции. Почему вычитание максимума не меняет результат, но необходимо численно?
2. Почему FlashAttention быстрее наивной реализации, хотя выполняет **больше** FLOPs в обратном проходе (пересчёт \(P\))?
3. Как размер блоков `BLOCK_M`, `BLOCK_N` ограничен размером SRAM (shared memory) и чем грозит слишком большой блок?
4. Почему блочная реализация на PyTorch в п. 4 медленнее наивной? Что именно даёт слияние операций в одно ядро?
5. Как изменится анализ roofline на фазе decode (один запрос, длинный кэш)? Какие оптимизации применяются там (FlashDecoding, split-K)?

Выводы: сделали flash-attention на PyTorch и на Triton для GPU. Все версии проверили на совпадение с эталоном. Потом замерили скорость и память, flash-attention версия на Python оказалась очень медленной, а SDPA и Triton работают быстро и экономят память. Ещё посчитали по roofline, что на обе схемы упираются в память, а не в вычисления. В конце сделали обратный проход с пересчётом и проверили, что градиенты совпадают с autograd.